# **Dataset**

*Canberra Metro Light Rail Transit (Australia)*

Supporting paper for the data: https://www.sciencedirect.com/science/article/pii/S2772586324000017#sec0006

Supporting paper for the model: https://www.sciencedirect.com/science/article/pii/S0957417422004717?casa_token=6XJQyQVS1BAAAAAA:i8BlohLXY8mgg8WOtJTy2CB96h83KMo9peZkBTs1khmAd_Ax2LphI2Cl92dfJYYAXo6rox0OLg 

**CSV and txt Files:**

- Canberra_Metro_Light_Rail_Transit_Feed_-_Trip_Updates__Historical_Archive__3M.csv

    - https://www.data.act.gov.au/Transport/Canberra-Metro-Light-Rail-Transit-Feed-Trip-Update/jxpp-4iiz/about_data

- stops.txt and stop_times.txt (detailed info)

    - https://www.transport.act.gov.au/__data/assets/pdf_file/0006/1259934/TCCS-GTFS-Implementation-Specification-v3.3-Bus-and-Light-Rail.pdf
    - https://gtfs.org/documentation/schedule/reference/

- Canberra_Metro_Light_Rail_Transit_Feed_-_Vehicle_Updates_15M.csv 

    - https://www.data.act.gov.au/Transport/Canberra-Metro-Light-Rail-Transit-Feed-Vehicle-Upd/92fy-xvmy

- Hourly Weather (2021-2022) from Open-meteo API

**Light Rail Transit stations:** 14

- Gungahlin Place

- Manning Clark North

- Mapleton Avenue

- Nullarbor Avenue

- Well Station Drive

- Sandford Street

- EPIC and Race Course

- Philip Avenue

- Swinden Street

- Dickson Interchange

- MacArthur Avenue

- Ipima Street

- Elouera Street

- Alinga Street

## **A. Import libraries**

In [3]:
!pip install requests

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import pandas as pd
import requests
from time import sleep
from datetime import timedelta
from sklearn.discriminant_analysis import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import numpy as np

## **B. Extract and Load Dataset**

In [9]:
trip_df = pd.read_csv("trip.csv")
trip_df.head()

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_76243/2403966219.py:1: DtypeWarning: Columns (0,2,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  trip_df = pd.read_csv("trip.csv")


,tripID,stopSequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,stopID,timestamp,GTFSRupdateID,updateDeleted,tripScheduleRelationship,arrivalUncertainty,depatureUncertainty,tripUpdateSchedulerelationship
0,303,6,0,2022 Nov 25 12:09:07 AM,0,1970 Jan 01 10:00:00 AM,8112,2022 Nov 24 02:18:30 PM,"29,927,972",NaN,SCHEDULED,0,0,SCHEDULED
1,303,5,0,2022 Nov 25 12:07:02 AM,0,2022 Nov 25 12:07:22 AM,8110,2022 Nov 24 02:18:30 PM,"29,927,972",NaN,SCHEDULED,0,0,SCHEDULED
2,303,4,0,2022 Nov 25 12:05:10 AM,0,2022 Nov 25 12:05:30 AM,8108,2022 Nov 24 02:18:30 PM,"29,927,972",NaN,SCHEDULED,0,0,SCHEDULED
3,303,3,0,2022 Nov 25 12:03:34 AM,0,2022 Nov 25 12:03:54 AM,8106,2022 Nov 24 02:18:30 PM,"29,927,972",NaN,SCHEDULED,0,0,SCHEDULED
4,303,2,0,2022 Nov 25 12:01:44 AM,0,2022 Nov 25 12:02:04 AM,8104,2022 Nov 24 02:18:30 PM,"29,927,972",NaN,SCHEDULED,0,0,SCHEDULED


In [14]:
vehicle_df = pd.read_csv("vehicle.csv", engine='python', on_bad_lines='skip')
vehicle_df.head()

,tripID,location,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,stopSequence,GTFSRupdateID,updateDeleted,latitude,longitude,bearing,currentStatus,stopID
0,218,POINT (-35.25265121459961 149.13340759277344),"279,646,759",2022 Nov 24 02:18:30 PM,RUNNING_SMOOTHLY,11,LRV11,LRV11,11,"29,927,935",NaN,-35.252651,149.133408,7.006292,IN_TRANSIT_TO,"8,122"
1,219,POINT (-35.214393615722656 149.14639282226562),"269,037,395",2022 Nov 24 02:18:30 PM,RUNNING_SMOOTHLY,6,LRV6,LRV6,6,"29,927,933",NaN,-35.214394,149.146393,14.105850,IN_TRANSIT_TO,"8,112"
2,88,POINT (-35.200252532958984 149.14930725097656),"275,165,938",2022 Nov 24 02:18:30 PM,RUNNING_SMOOTHLY,9,LRV9,LRV9,11,"29,927,934",NaN,-35.200253,149.149307,190.114120,STOPPED_AT,"8,109"
3,89,POINT (-35.25030517578125 149.13375854492188),"247,643,849",2022 Nov 24 02:18:30 PM,RUNNING_SMOOTHLY,10,LRV10,LRV10,6,"29,927,932",NaN,-35.250305,149.133759,185.781158,STOPPED_AT,"8,119"
4,218,POINT (-35.25132751464844 149.1336212158203),"279,646,603",2022 Nov 24 02:18:15 PM,RUNNING_SMOOTHLY,11,LRV11,LRV11,11,"29,927,660",NaN,-35.251328,149.133621,6.989918,IN_TRANSIT_TO,"8,122"


In [15]:
stops = pd.read_csv("stops.txt")
stops.head()

,stop_id,stop_name,stop_lat,stop_lon,stop_url,location_type,wheelchair_boarding,parent_station
0,8100,Gungahlin Place Platform 1,-35.185604,149.135487,https://cmet.com.au/light-rail-stops/gungahlin...,0,0,GGN
1,8101,Gungahlin Place Platform 2,-35.185673,149.135465,https://cmet.com.au/light-rail-stops/gungahlin...,0,0,GGN
2,GGN,Gungahlin Place,-35.185639,149.135481,https://cmet.com.au/light-rail-stops/gungahlin...,1,0,GGN
3,8104,Manning Clark Crescent Platform 1,-35.186961,149.143422,https://cmet.com.au/light-rail-stops/manning-c...,0,0,MCK
4,8105,Manning Clark Crescent Platform 2,-35.187036,149.143406,https://cmet.com.au/light-rail-stops/manning-c...,0,0,MCK


In [16]:
stop_times = pd.read_csv("stop_times.txt")
stop_times.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint
0,157,06:00:00,06:00:00,8100,1,Alinga Street,0,1,1
1,157,06:01:44,06:02:04,8104,2,Alinga Street,0,0,1
2,157,06:03:34,06:03:54,8106,3,Alinga Street,0,0,1
3,157,06:05:10,06:05:30,8108,4,Alinga Street,0,0,1
4,157,06:07:02,06:07:22,8110,5,Alinga Street,0,0,1


In [18]:
stations = [
    {"name": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"name": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"name": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"name": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"name": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"name": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"name": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"name": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"name": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"name": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"name": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"name": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"name": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"name": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

start = '2021-01-01'
end = '2021-12-31'

params = ",".join([
    "temperature_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "winddirection_10m"
])

all_data = []

for station in stations:
    print(f"Fetching hourly weather for {station['name']}...")

    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={station['lat']}&longitude={station['lon']}"
        f"&start_date={start}&end_date={end}"
        f"&hourly={params}"
        f"&timezone=Australia/Sydney"
    )

    response = requests.get(url)
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["station"] = station["name"]

    
    df["timestamp"] = pd.to_datetime(df["time"]).dt.strftime("%m/%d/%Y %I:%M:%S %p")
    df.drop(columns=["time"], inplace=True)

    all_data.append(df)
    sleep(1)

weather_hourly_df = pd.concat(all_data)
weather_hourly_df.to_csv("weather_2021.csv", index=False)

Fetching hourly weather for Gungahlin Place...
Fetching hourly weather for Manning Clark North...
Fetching hourly weather for Mapleton Avenue...
Fetching hourly weather for Nullarbor Avenue...
Fetching hourly weather for Well Station Drive...
Fetching hourly weather for Sandford Street...
Fetching hourly weather for EPIC & Racecourse...
Fetching hourly weather for Phillip Avenue...
Fetching hourly weather for Swinden Street...
Fetching hourly weather for Dickson Interchange...
Fetching hourly weather for Macarthur Avenue...
Fetching hourly weather for Ipima Street...
Fetching hourly weather for Elouera Street...
Fetching hourly weather for Alinga Street...


In [19]:
weather_2021_df = pd.read_csv("weather_2021.csv")
weather_2021_df.head()

,temperature_2m,apparent_temperature,precipitation,rain,snowfall,windspeed_10m,windgusts_10m,winddirection_10m,station,timestamp
0,14.6,13.2,0.0,0.0,0.0,11.6,22.3,112,Gungahlin Place,01/01/2021 12:00:00 AM
1,14.4,12.7,0.0,0.0,0.0,12.9,24.1,120,Gungahlin Place,01/01/2021 01:00:00 AM
2,14.2,12.3,0.0,0.0,0.0,13.5,25.2,124,Gungahlin Place,01/01/2021 02:00:00 AM
3,13.7,11.8,0.0,0.0,0.0,14.0,25.6,132,Gungahlin Place,01/01/2021 03:00:00 AM
4,13.5,11.5,0.0,0.0,0.0,14.3,26.6,133,Gungahlin Place,01/01/2021 04:00:00 AM


In [20]:
stations = [
    {"name": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"name": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"name": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"name": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"name": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"name": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"name": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"name": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"name": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"name": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"name": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"name": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"name": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"name": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

start = '2022-01-01'
end = '2022-12-31'

params = ",".join([
    "temperature_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "winddirection_10m"
])

all_data = []

for station in stations:
    print(f"Fetching hourly weather for {station['name']}...")

    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={station['lat']}&longitude={station['lon']}"
        f"&start_date={start}&end_date={end}"
        f"&hourly={params}"
        f"&timezone=Australia/Sydney"
    )

    response = requests.get(url)
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["station"] = station["name"]

    
    df["timestamp"] = pd.to_datetime(df["time"]).dt.strftime("%m/%d/%Y %I:%M:%S %p")
    df.drop(columns=["time"], inplace=True)

    all_data.append(df)
    sleep(1)

weather_hourly_df = pd.concat(all_data)
weather_hourly_df.to_csv("weather_2022.csv", index=False)

Fetching hourly weather for Gungahlin Place...
Fetching hourly weather for Manning Clark North...
Fetching hourly weather for Mapleton Avenue...
Fetching hourly weather for Nullarbor Avenue...
Fetching hourly weather for Well Station Drive...
Fetching hourly weather for Sandford Street...
Fetching hourly weather for EPIC & Racecourse...
Fetching hourly weather for Phillip Avenue...
Fetching hourly weather for Swinden Street...
Fetching hourly weather for Dickson Interchange...
Fetching hourly weather for Macarthur Avenue...
Fetching hourly weather for Ipima Street...
Fetching hourly weather for Elouera Street...
Fetching hourly weather for Alinga Street...


In [21]:
weather_2022_df = pd.read_csv("weather_2022.csv")
weather_2022_df.head()

,temperature_2m,apparent_temperature,precipitation,rain,snowfall,windspeed_10m,windgusts_10m,winddirection_10m,station,timestamp
0,14.2,14.4,0.0,0.0,0.0,5.3,10.4,118,Gungahlin Place,01/01/2022 12:00:00 AM
1,13.8,13.8,0.0,0.0,0.0,6.3,10.1,103,Gungahlin Place,01/01/2022 01:00:00 AM
2,13.5,13.4,0.0,0.0,0.0,6.2,11.2,80,Gungahlin Place,01/01/2022 02:00:00 AM
3,12.9,12.8,0.0,0.0,0.0,5.5,10.8,101,Gungahlin Place,01/01/2022 03:00:00 AM
4,12.3,12.2,0.0,0.0,0.0,5.2,9.7,115,Gungahlin Place,01/01/2022 04:00:00 AM


## **C. Data Cleaning and Preprocessing**

- Checking null values

- Dropping null values and irrelevant rows and/or columns

- Creating new column(s)

- Format correction and transforming categorical to numerical value

- Normalizing values

- Merging dataset

### *I. Checking null values*

In [22]:
trip_df.isnull().sum()

tripID                                  0
stopSequence                            0
arrivalDelay                            0
arrivalTime                             0
depatureDelay                           0
depatureTime                            0
stopID                                  0
timestamp                               0
GTFSRupdateID                           0
updateDeleted                     3537275
tripScheduleRelationship                0
arrivalUncertainty                      0
depatureUncertainty                     0
tripUpdateSchedulerelationship          0
dtype: int64

In [23]:
vehicle_df.isnull().sum()

tripID                       0
location                     0
odometer                     0
timestamp                    0
congestionLevel              0
vehicleID                    0
vehicleLabel                 0
vehicleLicenceplate          0
stopSequence                 0
GTFSRupdateID                0
updateDeleted          6642229
latitude                     0
longitude                    0
bearing                      0
currentStatus                0
stopID                       0
dtype: int64

In [24]:
stops.isnull().sum()

stop_id                0
stop_name              0
stop_lat               0
stop_lon               0
stop_url               0
location_type          0
wheelchair_boarding    0
parent_station         0
dtype: int64

In [25]:
stop_times.isnull().sum()

trip_id           0
arrival_time      0
departure_time    0
stop_id           0
stop_sequence     0
stop_headsign     0
pickup_type       0
drop_off_type     0
timepoint         0
dtype: int64

In [26]:
weather_2021_df.isnull().sum()

temperature_2m          0
apparent_temperature    0
precipitation           0
rain                    0
snowfall                0
windspeed_10m           0
windgusts_10m           0
winddirection_10m       0
station                 0
timestamp               0
dtype: int64

In [3]:
weather_2022_df = pd.read_csv("weather_2022.csv")
weather_2022_df.isnull().sum()

temperature_2m          0
apparent_temperature    0
precipitation           0
rain                    0
snowfall                0
windspeed_10m           0
windgusts_10m           0
winddirection_10m       0
station                 0
timestamp               0
dtype: int64

### *II. Dropping null values and irrelevant rows and/or columns*

### Trip Dataset

- filtering data from 2021-01-01 to 2022-11-25
- dropping NaN/Null values 
    - arrivalTime, depatureTime, and timestamp set to 1970

- dropping irrelevant columns 

    - timestamp

    - GTFSupdateID

    - updateDeleted

    - arrivalUncertainty

    - departureUncertainty

    - tripUpdateSchedulerelationship

In [28]:
trip_df = pd.read_csv("trip.csv")
trip_df['arrivalTime'] = pd.to_datetime(trip_df['arrivalTime'], errors='coerce')
trip_df['depatureTime'] = pd.to_datetime(trip_df['depatureTime'], errors='coerce')
trip_df['timestamp'] = pd.to_datetime(trip_df['timestamp'], errors='coerce')

trip_df = trip_df[
    (trip_df['arrivalTime'].dt.year > 2020) &
    (trip_df['depatureTime'].dt.year > 2020) &
    (trip_df['timestamp'].dt.year > 2020)
]

trip_df = trip_df.sort_values(by='arrivalTime')

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_76243/3506221594.py:1: DtypeWarning: Columns (0,2,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  trip_df = pd.read_csv("trip.csv")
/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_76243/3506221594.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  trip_df['arrivalTime'] = pd.to_datetime(trip_df['arrivalTime'], errors='coerce')
/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_76243/3506221594.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  trip_df['timestamp'] = pd.to_datetime(trip_df['timestamp'], errors='coerce')


In [29]:
columns_to_drop = [
    'timestamp',
    'GTFSRupdateID',
    'updateDeleted',
    'arrivalUncertainty',
    'depatureUncertainty',
    'tripUpdateSchedulerelationship'
]

trip_df = trip_df.drop(columns=columns_to_drop)

In [30]:
trip_df.head()

,tripID,stopSequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,stopID,tripScheduleRelationship
1962920,595,7,-39,2021-01-01 00:12:06,-56,2021-01-01 00:12:09,8116,SCHEDULED
1962919,447,8,-53,2021-01-01 00:12:07,-51,2021-01-01 00:12:29,8115,SCHEDULED
1962916,595,8,-51,2021-01-01 00:14:03,-44,2021-01-01 00:14:30,8118,SCHEDULED
1962914,595,9,-68,2021-01-01 00:15:36,-7,2021-01-01 00:16:57,8120,SCHEDULED
1962913,447,9,-35,2021-01-01 00:15:58,8,2021-01-01 00:17:01,8111,SCHEDULED


In [31]:
trip_df.tail()

,tripID,stopSequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,stopID,tripScheduleRelationship
7,152,13,0,2022-11-24 23:52:37,0,2022-11-24 23:52:57,8105,SCHEDULED
4,303,2,0,2022-11-25 00:01:44,0,2022-11-25 00:02:04,8104,SCHEDULED
3,303,3,0,2022-11-25 00:03:34,0,2022-11-25 00:03:54,8106,SCHEDULED
2,303,4,0,2022-11-25 00:05:10,0,2022-11-25 00:05:30,8108,SCHEDULED
1,303,5,0,2022-11-25 00:07:02,0,2022-11-25 00:07:22,8110,SCHEDULED


In [32]:
trip_df.to_csv("trip_v2.csv", index=False)

### Vehicle Dataset
- filtering data from 2021-01-01 to 2022-11-25
- renaming location col to currentLoc
- Dropping irrelevant columns

    - GTFSRupdateID
    - updateDeleted

- Dropping vehicle entries with no corresponding trip entry

In [34]:
vehicle_df = pd.read_csv("vehicle.csv", engine='python',on_bad_lines='skip')
vehicle_df["timestamp"] = pd.to_datetime(vehicle_df["timestamp"], errors="coerce")

vehicle_df = vehicle_df[
    (vehicle_df["timestamp"] >= "2021-01-01") &
    (vehicle_df["timestamp"] <= "2022-11-25")
]

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_76243/530150127.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  vehicle_df["timestamp"] = pd.to_datetime(vehicle_df["timestamp"], errors="coerce")


In [35]:
vehicle_df.rename(columns={'location': 'currentLoc'}, inplace=True)

In [36]:
columns_to_drop = [
    'GTFSRupdateID',
    'updateDeleted'
]
vehicle_df = vehicle_df.drop(columns=columns_to_drop)

In [37]:
vehicle_df.head()

,tripID,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,stopSequence,latitude,longitude,bearing,currentStatus,stopID
0,218,POINT (-35.25265121459961 149.13340759277344),"279,646,759",2022-11-24 14:18:30,RUNNING_SMOOTHLY,11,LRV11,LRV11,11,-35.252651,149.133408,7.006292,IN_TRANSIT_TO,"8,122"
1,219,POINT (-35.214393615722656 149.14639282226562),"269,037,395",2022-11-24 14:18:30,RUNNING_SMOOTHLY,6,LRV6,LRV6,6,-35.214394,149.146393,14.105850,IN_TRANSIT_TO,"8,112"
2,88,POINT (-35.200252532958984 149.14930725097656),"275,165,938",2022-11-24 14:18:30,RUNNING_SMOOTHLY,9,LRV9,LRV9,11,-35.200253,149.149307,190.114120,STOPPED_AT,"8,109"
3,89,POINT (-35.25030517578125 149.13375854492188),"247,643,849",2022-11-24 14:18:30,RUNNING_SMOOTHLY,10,LRV10,LRV10,6,-35.250305,149.133759,185.781158,STOPPED_AT,"8,119"
4,218,POINT (-35.25132751464844 149.1336212158203),"279,646,603",2022-11-24 14:18:15,RUNNING_SMOOTHLY,11,LRV11,LRV11,11,-35.251328,149.133621,6.989918,IN_TRANSIT_TO,"8,122"


In [38]:
vehicle_df.tail()

,tripID,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,stopSequence,latitude,longitude,bearing,currentStatus,stopID
6642224,49,POINT (-35.258853912353516 149.13238525390625),"154,385,587",2021-06-01 09:15:00,RUNNING_SMOOTHLY,9,LRV9,LRV9,5,-35.258854,149.132385,187.736420,IN_TRANSIT_TO,"8,121"
6642225,49,POINT (-35.2598762512207 149.13221740722656),"154,385,472",2021-06-01 09:14:45,RUNNING_SMOOTHLY,9,LRV9,LRV9,4,-35.259876,149.132217,181.639969,STOPPED_AT,"8,123"
6642226,183,POINT (-35.22325134277344 149.14410400390625),"137,402,285",2021-06-01 09:14:45,RUNNING_SMOOTHLY,2,LRV2,LRV2,6,-35.223251,149.144104,12.976830,IN_TRANSIT_TO,"8,114"
6642227,181,POINT (-35.25988006591797 149.13226318359375),"146,581,295",2021-06-01 09:14:45,RUNNING_SMOOTHLY,10,LRV10,LRV10,10,-35.259880,149.132263,7.002883,STOPPED_AT,"8,120"
6642228,184,POINT (-35.19819641113281 149.14988708496094),"156,876,200",2021-06-01 09:14:45,RUNNING_SMOOTHLY,11,LRV11,LRV11,4,-35.198196,149.149887,10.801080,IN_TRANSIT_TO,"8,108"


In [39]:
vehicle_df.to_csv("vehicle_df.csv", index=False)

In [41]:
vehicles = pd.read_csv("vehicle_df.csv", engine='python', quotechar='"', on_bad_lines='skip')
trips = pd.read_csv("trip_v2.csv", engine='python', quotechar='"', on_bad_lines='skip')

invalid_tripIDs = vehicles[~vehicles['tripID'].isin(trips['tripID'])]

total_vehicle_entries = vehicles.shape[0]
total_invalid_tripIDs = invalid_tripIDs.shape[0]
percentage_invalid = (total_invalid_tripIDs / total_vehicle_entries) * 100

print(f"Total vehicle entries: {total_vehicle_entries}")
print(f"Vehicle entries with tripID not in trip dataset: {total_invalid_tripIDs}")
print(f"Percentage invalid: {percentage_invalid:.2f}%")

display(invalid_tripIDs)

Total vehicle entries: 6642229
Vehicle entries with tripID not in trip dataset: 12056
Percentage invalid: 0.18%


,tripID,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,stopSequence,latitude,longitude,bearing,currentStatus,stopID
7258,4,POINT (-35.22099685668945 149.1448516845703),"231,377,961",2022-11-24 06:01:00,RUNNING_SMOOTHLY,5,LRV5,LRV5,0,-35.220997,149.144852,187.682785,IN_TRANSIT_TO,"8,113"
7260,4,POINT (-35.22179412841797 149.14466857910156),"231,377,872",2022-11-24 06:00:45,RUNNING_SMOOTHLY,5,LRV5,LRV5,0,-35.221794,149.144669,193.389297,STOPPED_AT,"8,113"
7266,4,POINT (-35.22209548950195 149.1445770263672),"231,377,841",2022-11-24 06:00:15,RUNNING_SMOOTHLY,5,LRV5,LRV5,1,-35.222095,149.144577,200.333603,IN_TRANSIT_TO,"8,113"
7268,4,POINT (-35.22283172607422 149.14418029785156),"231,377,746",2022-11-24 06:00:00,RUNNING_SMOOTHLY,5,LRV5,LRV5,1,-35.222832,149.144180,193.320129,IN_TRANSIT_TO,"8,113"
7269,4,POINT (-35.223453521728516 149.14349365234375),"231,377,638",2022-11-24 05:59:30,RUNNING_SMOOTHLY,5,LRV5,LRV5,1,-35.223454,149.143494,274.145050,IN_TRANSIT_TO,"8,113"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6629938,2,POINT (-35.21625518798828 149.14549255371094),"149,521,811",2021-06-02 05:29:45,RUNNING_SMOOTHLY,3,LRV3,LRV3,1,-35.216255,149.145493,0.000000,IN_TRANSIT_TO,"8,111"
6629939,2,POINT (-35.216278076171875 149.14549255371094),"149,521,809",2021-06-02 05:29:30,RUNNING_SMOOTHLY,3,LRV3,LRV3,1,-35.216278,149.145493,196.414474,IN_TRANSIT_TO,"8,111"
6629941,2,POINT (-35.217376708984375 149.1452178955078),"149,521,684",2021-06-02 05:29:00,RUNNING_SMOOTHLY,3,LRV3,LRV3,1,-35.217377,149.145218,183.739868,IN_TRANSIT_TO,"8,111"
6629943,2,POINT (-35.21931457519531 149.1450958251953),"149,521,472",2021-06-02 05:28:30,RUNNING_SMOOTHLY,3,LRV3,LRV3,1,-35.219315,149.145096,185.077591,IN_TRANSIT_TO,"8,111"


In [3]:
vehicles = pd.read_csv("vehicle_df.csv", engine='python', quotechar='"', on_bad_lines='skip')
trips = pd.read_csv("trip_v2.csv", engine='python', quotechar='"', on_bad_lines='skip')
vehicles_aligned = vehicles[vehicles['tripID'].isin(trips['tripID'])].copy()
print(f"Original vehicle entries: {vehicles.shape[0]}")
print(f"Vehicle entries after dropping invalid tripIDs: {vehicles_aligned.shape[0]}")


Original vehicle entries: 6642229
Vehicle entries after dropping invalid tripIDs: 6630173


In [4]:
vehicles_aligned.to_csv("vehicle_v2.csv", index=False)

### Stop and Stoptimes Dataset

- Dropping irrelevant columns

    - drop_off_type

    - location_type

    - wheelchair_boarding

    - parent_station

In [5]:
stops_df = pd.read_csv("stops.txt")
columns_to_drop = [
    'location_type',
    'wheelchair_boarding',
    'parent_station'
]
stops_df = stops_df.drop(columns=columns_to_drop)

In [6]:
stops_df.head()

,stop_id,stop_name,stop_lat,stop_lon,stop_url
0,8100,Gungahlin Place Platform 1,-35.185604,149.135487,https://cmet.com.au/light-rail-stops/gungahlin...
1,8101,Gungahlin Place Platform 2,-35.185673,149.135465,https://cmet.com.au/light-rail-stops/gungahlin...
2,GGN,Gungahlin Place,-35.185639,149.135481,https://cmet.com.au/light-rail-stops/gungahlin...
3,8104,Manning Clark Crescent Platform 1,-35.186961,149.143422,https://cmet.com.au/light-rail-stops/manning-c...
4,8105,Manning Clark Crescent Platform 2,-35.187036,149.143406,https://cmet.com.au/light-rail-stops/manning-c...


In [7]:
stops_df.to_csv("stops_v2.csv", index=False)

In [8]:
stop_times_df = pd.read_csv("stop_times.txt")
columns_to_drop = [
    'drop_off_type'
]
stop_times_df = stop_times_df.drop(columns=columns_to_drop)

In [9]:
stop_times_df.to_csv("stop_times_v2.csv", index=False)

### *III. Format correction and transforming categorical to numerical value* 

- **Format Correction**
    - IDs | float64 to int64
    - Numerical | features int64 to float64
    - Dates | object to date/datetime

- **Transforming categorical to numerical values**
    - tripScheduleRelationship(0-no_data, 1-scheduled, 2-skipped)
    - congestionLevel (0-unknown_congestion_level, 1-running_smoothly, 2-stop_and_go, 3-congestion, 4-severe_congestion)
    - currentStatus (0-stopped_at, 1-in_transit_to, 2-incoming_at)


### Trip Dataset

In [10]:
trip_df = pd.read_csv("trip_v2.csv")
trip_df.info()

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/3240473702.py:1: DtypeWarning: Columns (0,2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  trip_df = pd.read_csv("trip_v2.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1722628 entries, 0 to 1722627
Data columns (total 8 columns):
 #   Column                    Dtype 
---  ------                    ----- 
 0   tripID                    object
 1   stopSequence              int64 
 2   arrivalDelay              object
 3   arrivalTime               object
 4   depatureDelay             object
 5   depatureTime              object
 6   stopID                    int64 
 7   tripScheduleRelationship  object
dtypes: int64(2), object(6)
memory usage: 105.1+ MB


In [11]:
trip_df['arrivalTime'] = pd.to_datetime(trip_df['arrivalTime'])
trip_df['depatureTime'] = pd.to_datetime(trip_df['depatureTime'])

trip_df['tripScheduleRelationship'] = trip_df['tripScheduleRelationship'].replace({
    'NO_DATA': 0,
    'SCHEDULED': 1,
    'ADDED' : 2,
    'CANCELED' : 3,
    'SKIPPED': 4,
})

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/2718851934.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  trip_df['tripScheduleRelationship'] = trip_df['tripScheduleRelationship'].replace({


In [12]:
trip_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1722628 entries, 0 to 1722627
Data columns (total 8 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   tripID                    object        
 1   stopSequence              int64         
 2   arrivalDelay              object        
 3   arrivalTime               datetime64[ns]
 4   depatureDelay             object        
 5   depatureTime              datetime64[ns]
 6   stopID                    int64         
 7   tripScheduleRelationship  int64         
dtypes: datetime64[ns](2), int64(3), object(3)
memory usage: 105.1+ MB


In [13]:
trip_df.head()

,tripID,stopSequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,stopID,tripScheduleRelationship
0,595,7,-39,2021-01-01 00:12:06,-56,2021-01-01 00:12:09,8116,1
1,447,8,-53,2021-01-01 00:12:07,-51,2021-01-01 00:12:29,8115,1
2,595,8,-51,2021-01-01 00:14:03,-44,2021-01-01 00:14:30,8118,1
3,595,9,-68,2021-01-01 00:15:36,-7,2021-01-01 00:16:57,8120,1
4,447,9,-35,2021-01-01 00:15:58,8,2021-01-01 00:17:01,8111,1


In [14]:
trip_df.to_csv("trip_v3.csv", index=False)

### Vehicle Dataset

In [15]:
vehicle_df = pd.read_csv("vehicle_v2.csv")
vehicle_df.info()

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/552932293.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv("vehicle_v2.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6630173 entries, 0 to 6630172
Data columns (total 14 columns):
 #   Column               Dtype  
---  ------               -----  
 0   tripID               object 
 1   currentLoc           object 
 2   odometer             object 
 3   timestamp            object 
 4   congestionLevel      object 
 5   vehicleID            int64  
 6   vehicleLabel         object 
 7   vehicleLicenceplate  object 
 8   stopSequence         int64  
 9   latitude             float64
 10  longitude            float64
 11  bearing              float64
 12  currentStatus        object 
 13  stopID               object 
dtypes: float64(3), int64(2), object(9)
memory usage: 708.2+ MB


In [17]:
vehicle_df.head()

,tripID,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,stopSequence,latitude,longitude,bearing,currentStatus,stopID
0,218,POINT (-35.25265121459961 149.13340759277344),"279,646,759",2022-11-24 14:18:30,RUNNING_SMOOTHLY,11,LRV11,LRV11,11,-35.252651,149.133408,7.006292,IN_TRANSIT_TO,"8,122"
1,219,POINT (-35.214393615722656 149.14639282226562),"269,037,395",2022-11-24 14:18:30,RUNNING_SMOOTHLY,6,LRV6,LRV6,6,-35.214394,149.146393,14.105850,IN_TRANSIT_TO,"8,112"
2,88,POINT (-35.200252532958984 149.14930725097656),"275,165,938",2022-11-24 14:18:30,RUNNING_SMOOTHLY,9,LRV9,LRV9,11,-35.200253,149.149307,190.114120,STOPPED_AT,"8,109"
3,89,POINT (-35.25030517578125 149.13375854492188),"247,643,849",2022-11-24 14:18:30,RUNNING_SMOOTHLY,10,LRV10,LRV10,6,-35.250305,149.133759,185.781158,STOPPED_AT,"8,119"
4,218,POINT (-35.25132751464844 149.1336212158203),"279,646,603",2022-11-24 14:18:15,RUNNING_SMOOTHLY,11,LRV11,LRV11,11,-35.251328,149.133621,6.989918,IN_TRANSIT_TO,"8,122"


In [18]:
vehicle_df["stopID"] = (
    vehicle_df["stopID"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype("Int64")
)

In [19]:
vehicle_df.head()

,tripID,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,stopSequence,latitude,longitude,bearing,currentStatus,stopID
0,218,POINT (-35.25265121459961 149.13340759277344),"279,646,759",2022-11-24 14:18:30,RUNNING_SMOOTHLY,11,LRV11,LRV11,11,-35.252651,149.133408,7.006292,IN_TRANSIT_TO,8122
1,219,POINT (-35.214393615722656 149.14639282226562),"269,037,395",2022-11-24 14:18:30,RUNNING_SMOOTHLY,6,LRV6,LRV6,6,-35.214394,149.146393,14.105850,IN_TRANSIT_TO,8112
2,88,POINT (-35.200252532958984 149.14930725097656),"275,165,938",2022-11-24 14:18:30,RUNNING_SMOOTHLY,9,LRV9,LRV9,11,-35.200253,149.149307,190.114120,STOPPED_AT,8109
3,89,POINT (-35.25030517578125 149.13375854492188),"247,643,849",2022-11-24 14:18:30,RUNNING_SMOOTHLY,10,LRV10,LRV10,6,-35.250305,149.133759,185.781158,STOPPED_AT,8119
4,218,POINT (-35.25132751464844 149.1336212158203),"279,646,603",2022-11-24 14:18:15,RUNNING_SMOOTHLY,11,LRV11,LRV11,11,-35.251328,149.133621,6.989918,IN_TRANSIT_TO,8122


In [20]:
vehicle_df['timestamp'] = pd.to_datetime(vehicle_df['timestamp'])

# congestionLevel
vehicle_df['congestionLevel'] = vehicle_df['congestionLevel'].replace({
    'UNKNOWN_CONGESTION_LEVEL': 0,
    'RUNNING_SMOOTHLY': 1,
    'STOP_AND_GO': 2,
    'CONGESTION': 3,
    'SEVERE_CONGESTION': 4
})

# vehicleID
vehicle_df['vehicleID'] = vehicle_df['vehicleID'].fillna(0).astype('int64')

# stopSequence
vehicle_df['stopSequence'] = vehicle_df['stopSequence'].fillna(0).astype('int64')

# currentStatus
vehicle_df['currentStatus'] = vehicle_df['currentStatus'].replace({
    'STOPPED_AT': 0,
    'IN_TRANSIT_TO': 1,
    'INCOMING_AT': 2
})

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/874270791.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vehicle_df['congestionLevel'] = vehicle_df['congestionLevel'].replace({
/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/874270791.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vehicle_df['currentStatus'] = vehicle_df['currentStatus'].replace({


In [21]:
vehicle_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6630173 entries, 0 to 6630172
Data columns (total 14 columns):
 #   Column               Dtype         
---  ------               -----         
 0   tripID               object        
 1   currentLoc           object        
 2   odometer             object        
 3   timestamp            datetime64[ns]
 4   congestionLevel      int64         
 5   vehicleID            int64         
 6   vehicleLabel         object        
 7   vehicleLicenceplate  object        
 8   stopSequence         int64         
 9   latitude             float64       
 10  longitude            float64       
 11  bearing              float64       
 12  currentStatus        int64         
 13  stopID               Int64         
dtypes: Int64(1), datetime64[ns](1), float64(3), int64(4), object(5)
memory usage: 714.5+ MB


In [22]:
vehicle_df = vehicle_df.sort_values(by='timestamp')
vehicle_df.to_csv("vehicle_v3.csv", index=False)

In [7]:
vehicle_df = pd.read_csv("vehicle_v5.csv")
vehicle_df["odometer"] = (
    vehicle_df["odometer"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype("Int64")
)

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_93361/1374393581.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv("vehicle_v5.csv")


In [10]:
vehicle_df.to_csv("vehicle_v5.csv", index=False)

### **IV. Creating new columns (variables)**

### Trip Dataframe

- has_prev_stop
    - true: 1
    - false: 0
- dwell_time
- is_peak_hr
    - true: 1
    - false: 0

In [23]:
trip_df = pd.read_csv("trip_v3.csv")

trip_df.rename(columns={"stopID": "stop_id"}, inplace=True)
trip_df.rename(columns={"tripID": "trip_id"}, inplace=True)
trip_df.rename(columns={"stopSequence": "stop_sequence"}, inplace=True)

trip_df["trip_id"] = trip_df["trip_id"].astype(str)
trip_df["stop_id"] = trip_df["stop_id"].astype(str)
trip_df["stop_sequence"] = trip_df["stop_sequence"].astype(int)

stops_df = pd.read_csv("stops_v2.csv")
trip_df = pd.merge(trip_df, stops_df[['stop_id', 'stop_name']], on='stop_id', how='left')

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/2937924207.py:1: DtypeWarning: Columns (0,2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  trip_df = pd.read_csv("trip_v3.csv")


In [24]:
trip_df.head()

,trip_id,stop_sequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,stop_id,tripScheduleRelationship,stop_name
0,595,7,-39,2021-01-01 00:12:06,-56,2021-01-01 00:12:09,8116,1,Phillip Avenue Platform 1
1,447,8,-53,2021-01-01 00:12:07,-51,2021-01-01 00:12:29,8115,1,EPIC and Racecourse Platform 2
2,595,8,-51,2021-01-01 00:14:03,-44,2021-01-01 00:14:30,8118,1,Swinden Street Platform 1
3,595,9,-68,2021-01-01 00:15:36,-7,2021-01-01 00:16:57,8120,1,Dickson Platform 1
4,447,9,-35,2021-01-01 00:15:58,8,2021-01-01 00:17:01,8111,1,Well Station Drive Platform 2


#### Renaming column

In [25]:
trip_df.rename(columns={'stop_id': 'originStopID'}, inplace=True)
trip_df.rename(columns={'stop_name': 'originStopName'}, inplace=True)

In [26]:
trip_df.head()

,trip_id,stop_sequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,originStopID,tripScheduleRelationship,originStopName
0,595,7,-39,2021-01-01 00:12:06,-56,2021-01-01 00:12:09,8116,1,Phillip Avenue Platform 1
1,447,8,-53,2021-01-01 00:12:07,-51,2021-01-01 00:12:29,8115,1,EPIC and Racecourse Platform 2
2,595,8,-51,2021-01-01 00:14:03,-44,2021-01-01 00:14:30,8118,1,Swinden Street Platform 1
3,595,9,-68,2021-01-01 00:15:36,-7,2021-01-01 00:16:57,8120,1,Dickson Platform 1
4,447,9,-35,2021-01-01 00:15:58,8,2021-01-01 00:17:01,8111,1,Well Station Drive Platform 2


In [27]:
trip_df.to_csv("trip_v4.csv", index=False)

#### has previous stop

In [28]:
trip_df = pd.read_csv("trip_v4.csv")
trip_df = trip_df.sort_values(['trip_id', 'stop_sequence'])
trip_df['has_prev_stop'] = trip_df.groupby('trip_id')['originStopID'].shift(1).notnull().astype(int)

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/1162760544.py:1: DtypeWarning: Columns (0,2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  trip_df = pd.read_csv("trip_v4.csv")


In [29]:
trip_df.head()

,trip_id,stop_sequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,originStopID,tripScheduleRelationship,originStopName,has_prev_stop
461360,5,2,-6,2021-07-27 06:03:51,-9,2021-07-27 06:04:08,8109,1,Nullarbor Avenue Platform 2,0
464035,5,2,-9,2021-07-28 06:03:48,-11,2021-07-28 06:04:06,8109,1,Nullarbor Avenue Platform 2,1
466710,5,2,-20,2021-07-29 06:03:37,-24,2021-07-29 06:03:53,8109,1,Nullarbor Avenue Platform 2,1
475540,5,2,12,2021-08-02 06:04:09,11,2021-08-02 06:04:28,8109,1,Nullarbor Avenue Platform 2,1
478215,5,2,0,2021-08-03 06:03:57,0,2021-08-03 06:04:17,8109,1,Nullarbor Avenue Platform 2,1


#### Dwell time

In [30]:
trip_df['arrivalTime'] = pd.to_datetime(trip_df['arrivalTime'])
trip_df['depatureTime'] = pd.to_datetime(trip_df['depatureTime'])

trip_df['dwellTime_sec'] = (trip_df['depatureTime'] - trip_df['arrivalTime']).dt.total_seconds()

In [31]:
trip_df.head()

,trip_id,stop_sequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,originStopID,tripScheduleRelationship,originStopName,has_prev_stop,dwellTime_sec
461360,5,2,-6,2021-07-27 06:03:51,-9,2021-07-27 06:04:08,8109,1,Nullarbor Avenue Platform 2,0,17.0
464035,5,2,-9,2021-07-28 06:03:48,-11,2021-07-28 06:04:06,8109,1,Nullarbor Avenue Platform 2,1,18.0
466710,5,2,-20,2021-07-29 06:03:37,-24,2021-07-29 06:03:53,8109,1,Nullarbor Avenue Platform 2,1,16.0
475540,5,2,12,2021-08-02 06:04:09,11,2021-08-02 06:04:28,8109,1,Nullarbor Avenue Platform 2,1,19.0
478215,5,2,0,2021-08-03 06:03:57,0,2021-08-03 06:04:17,8109,1,Nullarbor Avenue Platform 2,1,20.0


#### Peak hour

In [32]:
trip_df['arrivalTime'] = pd.to_datetime(trip_df['arrivalTime'])

arrival_times = trip_df['arrivalTime'].dt.time

def is_peak(time):
    if (pd.to_datetime("07:00").time() <= time <= pd.to_datetime("09:00").time()) or \
        (pd.to_datetime("16:30").time() <= time <= pd.to_datetime("18:00").time()):
        return 1
    else:
        return 0

trip_df['is_peak_hour'] = arrival_times.apply(is_peak)

In [35]:
trip_df.head()

,trip_id,stop_sequence,arrivalDelay,arrivalTime,depatureDelay,depatureTime,originStopID,tripScheduleRelationship,originStopName,has_prev_stop,dwellTime_sec,is_peak_hour
461360,5,2,-6,2021-07-27 06:03:51,-9,2021-07-27 06:04:08,8109,1,Nullarbor Avenue Platform 2,0,17.0,0
464035,5,2,-9,2021-07-28 06:03:48,-11,2021-07-28 06:04:06,8109,1,Nullarbor Avenue Platform 2,1,18.0,0
466710,5,2,-20,2021-07-29 06:03:37,-24,2021-07-29 06:03:53,8109,1,Nullarbor Avenue Platform 2,1,16.0,0
475540,5,2,12,2021-08-02 06:04:09,11,2021-08-02 06:04:28,8109,1,Nullarbor Avenue Platform 2,1,19.0,0
478215,5,2,0,2021-08-03 06:03:57,0,2021-08-03 06:04:17,8109,1,Nullarbor Avenue Platform 2,1,20.0,0


In [36]:
trip_df.to_csv("trip_v5.csv", index=False)

### Vehicle Dataframe

- is_weekend
    - true: 1
    - false: 0

In [37]:
vehicle_df = pd.read_csv("vehicle_v3.csv")

vehicle_df.rename(columns={"stopID": "stop_id"}, inplace=True)
vehicle_df.rename(columns={"tripID": "trip_id"}, inplace=True)
vehicle_df.rename(columns={"stopSequence": "stop_sequence"}, inplace=True)

vehicle_df["trip_id"] = vehicle_df["trip_id"].astype(str)
vehicle_df["stop_id"] = vehicle_df["stop_id"].astype(str)
vehicle_df["stop_sequence"] = vehicle_df["stop_sequence"].astype(int)

stops_df = pd.read_csv("stops_v2.csv")
vehicle_df = pd.merge(vehicle_df, stops_df[['stop_id', 'stop_name']], on='stop_id', how='left')

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/4128043576.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv("vehicle_v3.csv")


In [38]:
vehicle_df.rename(columns={'stop_id': 'destinationStopID'}, inplace=True)

In [39]:
vehicle_df.rename(columns={'stop_name': 'destinationStopName'}, inplace=True)

In [40]:
vehicle_df.head()

,trip_id,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,stop_sequence,latitude,longitude,bearing,currentStatus,destinationStopID,destinationStopName
0,184,POINT (-35.19819641113281 149.14988708496094),"156,876,200",2021-06-01 09:14:45,1,11,LRV11,LRV11,4,-35.198196,149.149887,10.801080,1,8108,Nullarbor Avenue Platform 1
1,49,POINT (-35.2598762512207 149.13221740722656),"154,385,472",2021-06-01 09:14:45,1,9,LRV9,LRV9,4,-35.259876,149.132217,181.639969,0,8123,Macarthur Avenue Platform 2
2,181,POINT (-35.25988006591797 149.13226318359375),"146,581,295",2021-06-01 09:14:45,1,10,LRV10,LRV10,10,-35.259880,149.132263,7.002883,0,8120,Dickson Platform 1
3,183,POINT (-35.22325134277344 149.14410400390625),"137,402,285",2021-06-01 09:14:45,1,2,LRV2,LRV2,6,-35.223251,149.144104,12.976830,1,8114,EPIC and Racecourse Platform 1
4,184,POINT (-35.199607849121094 149.1495819091797),"156,876,351",2021-06-01 09:15:00,1,11,LRV11,LRV11,4,-35.199608,149.149582,10.481098,1,8108,Nullarbor Avenue Platform 1


In [41]:
vehicle_df.to_csv("vehicle_v4.csv", index=False)

#### Is weekend

In [42]:
vehicle_df = pd.read_csv("vehicle_v4.csv")
vehicle_df['timestamp'] = pd.to_datetime(vehicle_df['timestamp'])
vehicle_df['is_weekend'] = vehicle_df['timestamp'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_5905/3836564019.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv("vehicle_v4.csv")


In [44]:
vehicle_df.head()

,trip_id,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,stop_sequence,latitude,longitude,bearing,currentStatus,destinationStopID,destinationStopName,is_weekend
0,184,POINT (-35.19819641113281 149.14988708496094),"156,876,200",2021-06-01 09:14:45,1,11,LRV11,LRV11,4,-35.198196,149.149887,10.801080,1,8108,Nullarbor Avenue Platform 1,0
1,49,POINT (-35.2598762512207 149.13221740722656),"154,385,472",2021-06-01 09:14:45,1,9,LRV9,LRV9,4,-35.259876,149.132217,181.639969,0,8123,Macarthur Avenue Platform 2,0
2,181,POINT (-35.25988006591797 149.13226318359375),"146,581,295",2021-06-01 09:14:45,1,10,LRV10,LRV10,10,-35.259880,149.132263,7.002883,0,8120,Dickson Platform 1,0
3,183,POINT (-35.22325134277344 149.14410400390625),"137,402,285",2021-06-01 09:14:45,1,2,LRV2,LRV2,6,-35.223251,149.144104,12.976830,1,8114,EPIC and Racecourse Platform 1,0
4,184,POINT (-35.199607849121094 149.1495819091797),"156,876,351",2021-06-01 09:15:00,1,11,LRV11,LRV11,4,-35.199608,149.149582,10.481098,1,8108,Nullarbor Avenue Platform 1,0


In [43]:
vehicle_df.to_csv("vehicle_v5.csv", index=False)

### **V. Merging dataset**

### Weather 2021 and 2022

In [2]:
weather_2021_df = pd.read_csv("weather_2021.csv")
weather_2022_df = pd.read_csv("weather_2022.csv")


combined_weather_df = pd.concat([weather_2021_df, weather_2022_df], ignore_index=True)

combined_weather_df["timestamp"] = pd.to_datetime(combined_weather_df["timestamp"], errors="coerce")

combined_weather_df = combined_weather_df.sort_values(by="timestamp")

combined_weather_df.to_csv("merged_weather.csv", index=False)

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_3469/2799154388.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_weather_df["timestamp"] = pd.to_datetime(combined_weather_df["timestamp"], errors="coerce")


In [3]:
combined_weather_df.head()

,temperature_2m,apparent_temperature,precipitation,rain,snowfall,windspeed_10m,windgusts_10m,winddirection_10m,station,timestamp
0,14.6,13.2,0.0,0.0,0.0,11.6,22.3,112,Gungahlin Place,2021-01-01
105120,15.2,14.2,0.0,0.0,0.0,10.0,21.6,120,Elouera Street,2021-01-01
8760,14.6,13.2,0.0,0.0,0.0,11.6,22.3,112,Manning Clark North,2021-01-01
70080,15.2,14.2,0.0,0.0,0.0,10.0,21.6,120,Swinden Street,2021-01-01
26280,14.6,13.3,0.0,0.0,0.0,11.6,22.3,112,Nullarbor Avenue,2021-01-01


### Trip and Vehicle

In [4]:
trip = pd.read_csv("trip_v5.csv")

trip["arrivalTime"] = pd.to_datetime(trip["arrivalTime"], errors="coerce")

cutoff = pd.Timestamp("2021-06-01 09:13:40")

trip_filtered = trip[trip["arrivalTime"] >= cutoff]
trip_filtered = trip_filtered.sort_values(by="arrivalTime")

print(trip_filtered.head())

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_3617/3298522097.py:4: DtypeWarning: Columns (0,2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  trip = pd.read_csv("trip_v5.csv")


       trip_id  stop_sequence arrivalDelay         arrivalTime depatureDelay  \
340602      47              6           14 2021-06-01 09:13:40            24   
340603     184              3           11 2021-06-01 09:13:45            13   
340604      49              4          -16 2021-06-01 09:14:19            -9   
340605     181             10            2 2021-06-01 09:14:21            10   
340606      45             11          -57 2021-06-01 09:14:41           -53   

               depatureTime  originStopID  tripScheduleRelationship  \
340602  2021-06-01 09:14:10          8119                         1   
340603  2021-06-01 09:14:07          8106                         1   
340604  2021-06-01 09:14:46          8123                         1   
340605  2021-06-01 09:14:49          8122                         1   
340606  2021-06-01 09:15:05          8107                         1   

                     originStopName  has_prev_stop  dwellTime_sec  \
340602    Swinden Stree

In [5]:
trip_filtered.to_csv("trip_v6.csv", index=False)

In [ ]:
vehicle = pd.read_csv("vehicle_v5.csv", dtype={"trip_id": str}) 
trip = pd.read_csv("trip_v6.csv", dtype={"trip_id": str}) 

vehicle = vehicle.drop_duplicates(["trip_id", "stop_sequence", "timestamp"]) 
trip = trip.drop_duplicates(["trip_id", "stop_sequence", "arrivalTime", "depatureTime"]) 

vehicle = vehicle.rename(columns={"stop_sequence": "vehicle_stop_sequence"}) 
trip = trip.rename(columns={"stop_sequence": "trip_stop_sequence"}) 
trip = trip.rename(columns={"depatureDelay": "departureDelay"})
trip = trip.rename(columns={"depatureTime": "departureTime"})

vehicle['date'] = pd.to_datetime(vehicle['timestamp']).dt.date 
trip['date'] = pd.to_datetime(trip['arrivalTime']).dt.date 

# Compute the previous stop_sequence for merge 
trip['vehicle_stop_sequence'] = trip['trip_stop_sequence'] + 1 

# Merge with date check 
merged = pd.merge( vehicle, trip, on=['trip_id', 'vehicle_stop_sequence', 'date'], how='left' )

# keep all vehicle rows, fill with NaN if no previous stop on that date
merged = merged.drop(columns=['vehicle_stop_sequence_y', 'date'], errors='ignore') 
merged = merged.rename(columns={'vehicle_stop_sequence_x': 'vehicle_stop_sequence'}) 

column_order = [ "trip_id","currentLoc","odometer","timestamp","congestionLevel",
                "vehicleID","vehicleLabel","vehicleLicenceplate","vehicle_stop_sequence", "latitude",
                "longitude","bearing","currentStatus","destinationStopID", "destinationStopName",
                "is_weekend","trip_stop_sequence","arrivalDelay", "arrivalTime","departureDelay",
                "departureTime","originStopID", "tripScheduleRelationship","originStopName",
                "has_prev_stop", "dwellTime_sec","is_peak_hour" ] 

merged = merged[[col for col in column_order if col in merged.columns]] 
print("Merge complete. Rows:", len(merged)) 
merged.head()

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_59196/4065997664.py:2: DtypeWarning: Columns (2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  trip = pd.read_csv("trip_v6.csv", dtype={"trip_id": str})


Merge complete. Rows: 6716078


,trip_id,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,vehicle_stop_sequence,latitude,...,arrivalDelay,arrivalTime,departureDelay,departureTime,originStopID,tripScheduleRelationship,originStopName,has_prev_stop,dwellTime_sec,is_peak_hour
0,184,POINT (-35.19819641113281 149.14988708496094),156876200,2021-06-01 09:14:45,1,11,LRV11,LRV11,4,-35.198196,...,11,2021-06-01 09:13:45,13,2021-06-01 09:14:07,8106.0,1.0,Mapleton Avenue Platform 1,1.0,22.0,0.0
1,49,POINT (-35.2598762512207 149.13221740722656),154385472,2021-06-01 09:14:45,1,9,LRV9,LRV9,4,-35.259876,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,181,POINT (-35.25988006591797 149.13226318359375),146581295,2021-06-01 09:14:45,1,10,LRV10,LRV10,10,-35.259880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,183,POINT (-35.22325134277344 149.14410400390625),137402285,2021-06-01 09:14:45,1,2,LRV2,LRV2,6,-35.223251,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,184,POINT (-35.199607849121094 149.1495819091797),156876351,2021-06-01 09:15:00,1,11,LRV11,LRV11,4,-35.199608,...,11,2021-06-01 09:13:45,13,2021-06-01 09:14:07,8106.0,1.0,Mapleton Avenue Platform 1,1.0,22.0,0.0


In [3]:
merged.to_csv("merged.csv", index=False)

In [4]:
merged = pd.read_csv("merged.csv")

key_cols = [
    "timestamp",
    "currentLoc",
    "vehicle_stop_sequence",
    "destinationStopName",
    "trip_stop_sequence",
    "arrivalDelay",
    "arrivalTime",
    "departureDelay",
    "departureTime",
    "originStopName"
]

merged[key_cols].head(30)

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_59196/1269374728.py:2: DtypeWarning: Columns (0,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv("merged.csv")


,timestamp,currentLoc,vehicle_stop_sequence,destinationStopName,trip_stop_sequence,arrivalDelay,arrivalTime,departureDelay,departureTime,originStopName
0,2021-06-01 09:14:45,POINT (-35.19819641113281 149.14988708496094),4,Nullarbor Avenue Platform 1,3.0,11.0,2021-06-01 09:13:45,13.0,2021-06-01 09:14:07,Mapleton Avenue Platform 1
1,2021-06-01 09:14:45,POINT (-35.2598762512207 149.13221740722656),4,Macarthur Avenue Platform 2,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-06-01 09:14:45,POINT (-35.25988006591797 149.13226318359375),10,Dickson Platform 1,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-06-01 09:14:45,POINT (-35.22325134277344 149.14410400390625),6,EPIC and Racecourse Platform 1,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-06-01 09:15:00,POINT (-35.199607849121094 149.1495819091797),4,Nullarbor Avenue Platform 1,3.0,11.0,2021-06-01 09:13:45,13.0,2021-06-01 09:14:07,Mapleton Avenue Platform 1
5,2021-06-01 09:15:00,POINT (-35.24325942993164 149.1348876953125),8,Phillip Avenue Platform 1,NaN,NaN,NaN,NaN,NaN,NaN
6,2021-06-01 09:15:00,POINT (-35.2607307434082 149.1321258544922),11,Macarthur Avenue Platform 1,10.0,2.0,2021-06-01 09:14:21,10.0,2021-06-01 09:14:49,Macarthur Avenue Platform 1
7,2021-06-01 09:15:00,POINT (-35.258853912353516 149.13238525390625),5,Dickson Platform 2,4.0,-16.0,2021-06-01 09:14:19,-9.0,2021-06-01 09:14:46,Macarthur Avenue Platform 2
8,2021-06-01 09:15:00,POINT (-35.19356155395508 149.1509246826172),11,Nullarbor Avenue Platform 2,NaN,NaN,NaN,NaN,NaN,NaN
9,2021-06-01 09:15:00,POINT (-35.22120666503906 149.1448211669922),9,Well Station Drive Platform 2,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
missing_trip_data_count = merged[key_cols].isna().any(axis=1).sum()
print("Rows with missing trip data:", missing_trip_data_count)

same_origin_destination_count = (merged['originStopName'] == merged['destinationStopName']).sum()
print("Rows with same origin and destination:", same_origin_destination_count)

merged['arrivalDelay'] = pd.to_numeric(merged['arrivalDelay'], errors='coerce')
merged['departureDelay'] = pd.to_numeric(merged['departureDelay'], errors='coerce')

positive_delay_count = ((merged['arrivalDelay'] > 0) | (merged['departureDelay'] > 0)).sum()
print("Rows with positive arrival or departure delay:", positive_delay_count)

negative_delay_count = ((merged['arrivalDelay'] < 0) | (merged['departureDelay'] < 0)).sum()
print("Rows with negative arrival or departure delay:", negative_delay_count)

zero_delay_count = ((merged['arrivalDelay'] == 0) & (merged['departureDelay'] == 0)).sum()
print("Rows with zero arrival and departure delay:", zero_delay_count)

for col in ['timestamp', 'arrivalTime', 'departureTime']:
    merged[col] = pd.to_datetime(merged[col], errors='coerce')

time_threshold = timedelta(minutes=5)

# vehicleStopSequence > stopSequence
valid_sequence = merged['vehicle_stop_sequence'] > merged['trip_stop_sequence']

# Different origin and destination
different_stops = merged['originStopName'] != merged['destinationStopName']

# Timestamp is close to arrival or departure time
timestamp_near_time = (
    (abs(merged['timestamp'] - merged['arrivalTime']) <= time_threshold) |
    (abs(merged['timestamp'] - merged['departureTime']) <= time_threshold)
)

# Combine all checks
valid_rows = valid_sequence & different_stops & timestamp_near_time

# Count valid and invalid rows
valid_rows_count = valid_rows.sum()
invalid_rows_count = len(merged) - valid_rows_count

print("Valid rows (sequence > stop, different stops, close timestamp):", valid_rows_count)
print("Invalid rows (failed one or more conditions):", invalid_rows_count)


Rows with missing trip data: 643919
Rows with same origin and destination: 467713
Rows with positive arrival or departure delay: 3871611
Rows with negative arrival or departure delay: 2843567
Rows with zero arrival and departure delay: 102086
Valid rows (sequence > stop, different stops, close timestamp): 5601968
Invalid rows (failed one or more conditions): 1114110


In [ ]:
# Filter only valid rows
cleaned_df = merged[valid_rows].copy()
cleaned_df.to_csv("merged_v2.csv", index=False)

#### Cleaning and creating new columns

In [55]:
df = pd.read_csv("merged_v2.csv")

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_59196/181747703.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("merged_v2.csv")


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5601968 entries, 0 to 5601967
Data columns (total 27 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   trip_id                   object 
 1   currentLoc                object 
 2   odometer                  int64  
 3   timestamp                 object 
 4   congestionLevel           int64  
 5   vehicleID                 int64  
 6   vehicleLabel              object 
 7   vehicleLicenceplate       object 
 8   vehicle_stop_sequence     int64  
 9   latitude                  float64
 10  longitude                 float64
 11  bearing                   float64
 12  currentStatus             int64  
 13  destinationStopID         int64  
 14  destinationStopName       object 
 15  is_weekend                int64  
 16  trip_stop_sequence        int64  
 17  arrivalDelay              float64
 18  arrivalTime               object 
 19  departureDelay            float64
 20  departureTime           

In [57]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['arrivalTime'] = pd.to_datetime(df['arrivalTime'], errors='coerce')
df['departureTime'] = pd.to_datetime(df['departureTime'], errors='coerce')

for col in [
    'trip_stop_sequence', 'originStopID',
    'tripScheduleRelationship', 'has_prev_stop',
    'is_peak_hour'
]:
    df[col] = df[col].astype('Int64')

for col in ['arrivalDelay', 'departureDelay']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5601968 entries, 0 to 5601967
Data columns (total 27 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   trip_id                   object        
 1   currentLoc                object        
 2   odometer                  int64         
 3   timestamp                 datetime64[ns]
 4   congestionLevel           int64         
 5   vehicleID                 int64         
 6   vehicleLabel              object        
 7   vehicleLicenceplate       object        
 8   vehicle_stop_sequence     int64         
 9   latitude                  float64       
 10  longitude                 float64       
 11  bearing                   float64       
 12  currentStatus             int64         
 13  destinationStopID         int64         
 14  destinationStopName       object        
 15  is_weekend                int64         
 16  trip_stop_sequence        Int64         
 17  arrivalD

In [59]:
earliest =df["timestamp"].min()
latest = df["timestamp"].max()

print("Earliest date:", earliest)
print("Latest date:", latest)

Earliest date: 2021-06-01 09:14:45
Latest date: 2022-11-24 14:18:30


In [60]:
df.to_csv("merged_v2.csv", index=False)

### Adding travel time, distance, and speed

travel time: departure time - timestamp

distance: current odometer - previous odometer

speed: distance / travel time

speed_kph: speed_mps * 3.6

In [4]:
df = pd.read_csv("merged_v2.csv")
df = df.sort_values(by='timestamp')

df.drop(columns=['has_prev_stop'], inplace=True, errors='ignore')

df['has_prev_stop'] = (
    df
    .groupby('trip_id')['originStopID']
    .shift(1)
    .notnull()
    .astype(int)
)

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_36295/2319524326.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("merged_v2.csv")


In [5]:
df.head()

,trip_id,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,vehicle_stop_sequence,latitude,...,arrivalDelay,arrivalTime,departureDelay,departureTime,originStopID,tripScheduleRelationship,originStopName,dwellTime_sec,is_peak_hour,has_prev_stop
0,184,POINT (-35.19819641113281 149.14988708496094),156876200,2021-06-01 09:14:45,1,11,LRV11,LRV11,4,-35.198196,...,11.0,2021-06-01 09:13:45,13.0,2021-06-01 09:14:07,8106,1,Mapleton Avenue Platform 1,22.0,0,0
1,184,POINT (-35.199607849121094 149.1495819091797),156876351,2021-06-01 09:15:00,1,11,LRV11,LRV11,4,-35.199608,...,11.0,2021-06-01 09:13:45,13.0,2021-06-01 09:14:07,8106,1,Mapleton Avenue Platform 1,22.0,0,1
2,49,POINT (-35.258853912353516 149.13238525390625),154385587,2021-06-01 09:15:00,1,9,LRV9,LRV9,5,-35.258854,...,-16.0,2021-06-01 09:14:19,-9.0,2021-06-01 09:14:46,8123,1,Macarthur Avenue Platform 2,27.0,0,0
3,47,POINT (-35.23988342285156 149.13803100585938),137985437,2021-06-01 09:15:00,1,13,LRV13,LRV13,7,-35.239883,...,14.0,2021-06-01 09:13:40,24.0,2021-06-01 09:14:10,8119,1,Swinden Street Platform 2,30.0,0,0
4,184,POINT (-35.200260162353516 149.1494140625),156876426,2021-06-01 09:15:30,1,11,LRV11,LRV11,4,-35.200260,...,11.0,2021-06-01 09:13:45,13.0,2021-06-01 09:14:07,8106,1,Mapleton Avenue Platform 1,22.0,0,1


In [ ]:
MAX_SPEED_KPH = 80  # anything > 80 becomes null (>=81)


df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['departureTime'] = pd.to_datetime(df['departureTime'], errors='coerce')

# No 3am cutoff
df['service_date'] = (df['timestamp']).dt.date

# Global time-stamp
df = df.sort_values('timestamp').reset_index(drop=True)
df['_orig_order'] = df.index

g = ['trip_id', 'service_date']

# temporary sort for correct groupwise computations
df_tmp = df.sort_values(g + ['timestamp']).copy()

# has_prev_stop within trip_id+service_date
df_tmp['has_prev_stop'] = (
    df_tmp.groupby(g)['originStopID'].shift(1).notna().astype(int)
)

# Distance (odometer delta)
df_tmp['prev_odometer'] = df_tmp.groupby(g)['odometer'].shift(1)
df_tmp['distance_m'] = df_tmp['odometer'] - df_tmp['prev_odometer']
df_tmp.loc[df_tmp['has_prev_stop'] == 0, 'distance_m'] = pd.NA
df_tmp.loc[df_tmp['distance_m'].notna() & (df_tmp['distance_m'] < 0), 'distance_m'] = pd.NA

# Time delta
df_tmp['prev_timestamp'] = df_tmp.groupby(g)['timestamp'].shift(1)
df_tmp['delta_t_sec'] = (df_tmp['timestamp'] - df_tmp['prev_timestamp']).dt.total_seconds()
df_tmp.loc[(df_tmp['has_prev_stop'] == 0) | (df_tmp['delta_t_sec'] <= 0), 'delta_t_sec'] = pd.NA

# Speed
df_tmp['speed_mps'] = pd.NA
mask = df_tmp['distance_m'].notna() & df_tmp['delta_t_sec'].notna() & (df_tmp['delta_t_sec'] > 0)
df_tmp.loc[mask, 'speed_mps'] = df_tmp.loc[mask, 'distance_m'] / df_tmp.loc[mask, 'delta_t_sec']
df_tmp['speed_kph'] = df_tmp['speed_mps'] * 3.6

# Travel time since station departure -- handles predaparture 218 was set based on the dataset (the maximum observed negative delta_t_sec was -218)
tol = 218  # seconds
dt = (df_tmp['timestamp'] - df_tmp['departureTime']).dt.total_seconds()
df_tmp['travel_time_sec'] = dt
df_tmp.loc[(dt >= -tol) & (dt < 0), 'travel_time_sec'] = 0
df_tmp.loc[dt < -tol, 'travel_time_sec'] = pd.NA


# Prep vehicle2 once (shared by both fallback passes) --  for getting the raw previous data
TOLERANCE = pd.Timedelta(hours=3)

v2 = pd.read_csv("vehicle2.csv")

if 'tripID' in v2.columns and 'trip_id' not in v2.columns:
    v2 = v2.rename(columns={'tripID': 'trip_id'})

v2['timestamp'] = pd.to_datetime(v2['timestamp'], errors='coerce')

# Clean odometer strings like "125,678,654" -- for raw dataset handling
v2['odometer'] = (
    v2['odometer']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
v2['odometer'] = pd.to_numeric(v2['odometer'], errors='coerce')

v2 = v2[v2['trip_id'].notna() & v2['timestamp'].notna() & v2['odometer'].notna()].copy()
v2['service_date'] = (v2['timestamp'] - pd.Timedelta(hours=3)).dt.date
v2 = v2[['trip_id', 'service_date', 'timestamp', 'odometer']].copy()

# Rename RHS columns so we can keep them after merge
v2 = v2.rename(columns={
    'timestamp': 'v2_prev_timestamp',
    'odometer': 'v2_prev_odometer'
})
v2['timestamp'] = v2['v2_prev_timestamp']  # join key copy for merge_asof

# merge_asof needs global timestamp sort
v2 = v2.sort_values(['timestamp', 'trip_id', 'service_date']).reset_index(drop=True)


# Fallback pass #1: overwrite for has_prev_stop==0
df_tmp['used_vehicle2_prev'] = 0

cand1 = df_tmp[df_tmp['has_prev_stop'] == 0].copy()
cand1 = cand1[cand1['trip_id'].notna() & cand1['timestamp'].notna() & cand1['odometer'].notna()].copy()
cand1 = cand1.reset_index().rename(columns={'index': '_df_tmp_idx'})
cand1 = cand1.sort_values(['timestamp', 'trip_id', 'service_date']).reset_index(drop=True)

matched1 = pd.merge_asof(
    cand1,
    v2,
    on='timestamp',
    by=['trip_id', 'service_date'],
    direction='backward',
    allow_exact_matches=False,
    tolerance=TOLERANCE
)

matched1['distance_m_from_v2'] = matched1['odometer'] - matched1['v2_prev_odometer']
matched1['delta_t_sec_from_v2'] = (matched1['timestamp'] - matched1['v2_prev_timestamp']).dt.total_seconds()

valid1 = (
    matched1['v2_prev_odometer'].notna() &
    matched1['v2_prev_timestamp'].notna() &
    (matched1['delta_t_sec_from_v2'] > 0) &
    (matched1['distance_m_from_v2'] >= 0)
)

matched1.loc[valid1, 'speed_mps_from_v2'] = matched1.loc[valid1, 'distance_m_from_v2'] / matched1.loc[valid1, 'delta_t_sec_from_v2']
matched1.loc[valid1, 'speed_kph_from_v2'] = matched1.loc[valid1, 'speed_mps_from_v2'] * 3.6

# Apply max-speed rule to fallback computed speeds BEFORE writing back
valid1 = valid1 & matched1['speed_kph_from_v2'].notna() & (matched1['speed_kph_from_v2'] <= MAX_SPEED_KPH)

fill_rows1 = matched1.loc[valid1, '_df_tmp_idx']
df_tmp.loc[fill_rows1, 'distance_m'] = matched1.loc[valid1, 'distance_m_from_v2'].values
df_tmp.loc[fill_rows1, 'delta_t_sec'] = matched1.loc[valid1, 'delta_t_sec_from_v2'].values
df_tmp.loc[fill_rows1, 'speed_mps'] = matched1.loc[valid1, 'speed_mps_from_v2'].values
df_tmp.loc[fill_rows1, 'speed_kph'] = matched1.loc[valid1, 'speed_kph_from_v2'].values
df_tmp.loc[fill_rows1, 'used_vehicle2_prev'] = 1

print("Filled from vehicle2 for has_prev_stop==0:", int(df_tmp['used_vehicle2_prev'].sum()))


# max-speed enforcement on the whole dataset
# "beyond 81 should be null" => speed_kph > 80 -> set null
over = df_tmp['speed_kph'].notna() & (df_tmp['speed_kph'] > MAX_SPEED_KPH)
df_tmp.loc[over, ['speed_kph', 'speed_mps', 'distance_m', 'delta_t_sec']] = pd.NA

event_key = [
    'trip_id',
    'service_date',
    'timestamp',
    'odometer',
    'vehicle_stop_sequence'
]

df_tmp['_delta_ok'] = (df_tmp['delta_t_sec'].notna() & (df_tmp['delta_t_sec'] > 0)).astype(int)
df_tmp['_dist_ok']  = (df_tmp['distance_m'].notna() & (df_tmp['distance_m'] > 0)).astype(int)

# Restore original order
df_out = (
    df_tmp
    .sort_values('_orig_order')
    .drop_duplicates(subset=event_key, keep='first')
    .drop(columns=['_orig_order', 'prev_odometer', 'prev_timestamp', 'used_vehicle2_prev'])
    .reset_index(drop=True)
) 

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_36295/3877370179.py:55: DtypeWarning: Columns (0,9,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  v2 = pd.read_csv("vehicle2.csv")
/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_36295/3877370179.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  v2['timestamp'] = pd.to_datetime(v2['timestamp'], errors='coerce')


Filled from vehicle2 for has_prev_stop==0: 41037


In [21]:
df_out.drop(columns=['_delta_ok', '_dist_ok'], inplace=True)

In [22]:
df_out.isna().sum()

trip_id                         0
currentLoc                      0
odometer                        0
timestamp                       0
congestionLevel                 0
vehicleID                       0
vehicleLabel                    0
vehicleLicenceplate             0
vehicle_stop_sequence           0
latitude                        0
longitude                       0
bearing                         0
currentStatus                   0
destinationStopID               0
destinationStopName             0
is_weekend                      0
trip_stop_sequence              0
arrivalDelay                  351
arrivalTime                     0
departureDelay                360
departureTime                   0
originStopID                    0
tripScheduleRelationship        0
originStopName                  0
dwellTime_sec                   0
is_peak_hour                    0
has_prev_stop                   0
service_date                    0
distance_m                  78706
delta_t_sec   

In [23]:
df_out.to_csv("merged_v3.csv", index=False)

### Adding slowdown

In [24]:
df = pd.read_csv("merged_v3.csv")

stations = [
    {"name": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"name": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"name": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"name": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"name": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"name": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"name": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"name": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"name": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"name": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"name": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"name": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"name": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"name": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

# Extract the first word from each station name and stop name
station_coords_first = {s['name'].split()[0].lower(): (s['lat'], s['lon']) for s in stations}

def map_station_firstword(stop_name):
    if pd.isna(stop_name):
        return (None, None)
    first_word = stop_name.split()[0].lower()
    return station_coords_first.get(first_word, (None, None))

# Apply to origin and destination
df['originLat'], df['originLon'] = zip(*df['originStopName'].map(map_station_firstword))
df['destinationLat'], df['destinationLon'] = zip(*df['destinationStopName'].map(map_station_firstword))

LOW_SPEED_THRESHOLD = 6.94  # m/s 25kmh * 0.27778
LONG_DWELL_THRESHOLD = 60  # sec
LONG_DELAY_THRESHOLD = 60  # sec

df['is_slow_speed'] = df['speed_mps'] < LOW_SPEED_THRESHOLD # use mps
df['is_long_dwell'] = df['dwellTime_sec'] > LONG_DWELL_THRESHOLD
df['is_delayed'] = (df['arrivalDelay'] > LONG_DELAY_THRESHOLD) | (df['departureDelay'] > LONG_DELAY_THRESHOLD)
df['is_congested'] = df['congestionLevel'] >= 3

df['is_slowdown'] = df[['is_slow_speed', 'is_long_dwell', 'is_delayed', 'is_congested']].any(axis=1)

df['slowdown_lat'] = None
df['slowdown_lon'] = None

mask = df['is_slowdown']
df.loc[mask, 'slowdown_lat'] = df.loc[mask, 'latitude']
df.loc[mask, 'slowdown_lon'] = df.loc[mask, 'longitude']


df['segment'] = df['originStopName'] + " → " + df['destinationStopName']
df['segment'] = df.apply(lambda row: row['segment'] if row['is_slowdown'] else 'NO SLOWDOWN', axis=1)

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_36295/1936706444.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("merged_v3.csv")


In [25]:
# Replace NaN with 0 for numeric columns
# numeric_cols = ['travel_time_sec', 'distance_m', 'speed_mps', 'speed_kph']
# df[numeric_cols] = df[numeric_cols].where(pd.notna(df[numeric_cols]), np.nan)

# Convert boolean columns to 1/0
bool_cols = ['is_slow_speed', 'is_long_dwell', 'is_delayed', 'is_congested', 'is_slowdown']
df[bool_cols] = df[bool_cols].astype(int)

df.head()

,trip_id,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,vehicle_stop_sequence,latitude,...,destinationLat,destinationLon,is_slow_speed,is_long_dwell,is_delayed,is_congested,is_slowdown,slowdown_lat,slowdown_lon,segment
0,184,POINT (-35.19819641113281 149.14988708496094),156876200,2021-06-01 09:14:45,1,11,LRV11,LRV11,4,-35.198196,...,-35.200556,149.149305,0,0,0,0,0,None,None,NO SLOWDOWN
1,184,POINT (-35.199607849121094 149.1495819091797),156876351,2021-06-01 09:15:00,1,11,LRV11,LRV11,4,-35.199608,...,-35.200556,149.149305,0,0,0,0,0,None,None,NO SLOWDOWN
2,49,POINT (-35.258853912353516 149.13238525390625),154385587,2021-06-01 09:15:00,1,9,LRV9,LRV9,5,-35.258854,...,-35.250586,149.133669,0,0,0,0,0,None,None,NO SLOWDOWN
3,47,POINT (-35.23988342285156 149.13803100585938),137985437,2021-06-01 09:15:00,1,13,LRV13,LRV13,7,-35.239883,...,-35.235895,149.143757,0,0,0,0,0,None,None,NO SLOWDOWN
4,184,POINT (-35.200260162353516 149.1494140625),156876426,2021-06-01 09:15:30,1,11,LRV11,LRV11,4,-35.200260,...,-35.200556,149.149305,1,0,0,0,1,-35.20026,149.149414,Mapleton Avenue Platform 1 → Nullarbor Avenue ...


In [27]:
df.to_csv("merged_v4.csv", index=False)

#### Weather and merged data

In [30]:
trips = pd.read_csv("merged_v4.csv")
weather = pd.read_csv("merged_weather.csv")

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_36295/3315926533.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv("merged_v4.csv")


In [ ]:
trips["timestamp"] = pd.to_datetime(trips["timestamp"], errors="coerce")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

trips["station_first"] = trips["originStopName"].fillna("").str.split().str[0].str.lower()
weather["station_first"] = weather["station"].fillna("").str.split().str[0].str.lower()

weather = weather.drop_duplicates(subset=["station_first", "timestamp"], keep="last")

trips["_original_index"] = trips.index

trips2 = trips.copy()
weather2 = weather.copy()

trips2["timestamp"] = pd.to_datetime(trips2["timestamp"], errors="coerce", utc=False)
weather2["timestamp"] = pd.to_datetime(weather2["timestamp"], errors="coerce", utc=False)

trips2["station_first"] = trips2["station_first"].astype(str)
weather2["station_first"] = weather2["station_first"].astype(str)

trips2 = trips2.dropna(subset=["station_first", "timestamp"]).copy()
weather2 = weather2.dropna(subset=["station_first", "timestamp"]).copy()

trips_sorted  = trips2.sort_values(["timestamp", "station_first"], kind="mergesort").reset_index(drop=True)
weather_sorted = weather2.sort_values(["timestamp", "station_first"], kind="mergesort").reset_index(drop=True)

merged_sorted = pd.merge_asof(
    trips_sorted,
    weather_sorted,
    left_on="timestamp",
    right_on="timestamp",
    by="station_first",
    direction="backward",
    allow_exact_matches=True
)

merged = (merged_sorted
          .sort_values("_original_index")
          .drop(columns=["station_first", "station"])
          .reset_index(drop=True))

merged.head()

,trip_id,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,vehicle_stop_sequence,latitude,...,segment,_original_index,temperature_2m,apparent_temperature,precipitation,rain,snowfall,windspeed_10m,windgusts_10m,winddirection_10m
0,184,POINT (-35.19819641113281 149.14988708496094),156876200,2021-06-01 09:14:45,1,11,LRV11,LRV11,4,-35.198196,...,NO SLOWDOWN,0,2.8,0.5,0.0,0.0,0.0,2.1,9.0,329
1,184,POINT (-35.199607849121094 149.1495819091797),156876351,2021-06-01 09:15:00,1,11,LRV11,LRV11,4,-35.199608,...,NO SLOWDOWN,1,2.8,0.5,0.0,0.0,0.0,2.1,9.0,329
2,49,POINT (-35.258853912353516 149.13238525390625),154385587,2021-06-01 09:15:00,1,9,LRV9,LRV9,5,-35.258854,...,NO SLOWDOWN,2,1.8,-0.6,0.0,0.0,0.0,2.5,12.2,82
3,47,POINT (-35.23988342285156 149.13803100585938),137985437,2021-06-01 09:15:00,1,13,LRV13,LRV13,7,-35.239883,...,NO SLOWDOWN,3,1.8,-0.6,0.0,0.0,0.0,2.5,12.2,82
4,184,POINT (-35.200260162353516 149.1494140625),156876426,2021-06-01 09:15:30,1,11,LRV11,LRV11,4,-35.200260,...,Mapleton Avenue Platform 1 → Nullarbor Avenue ...,4,2.8,0.5,0.0,0.0,0.0,2.1,9.0,329


In [ ]:
merged.drop(columns=["_original_index"], inplace=True)

In [36]:
merged.isna().sum()

trip_id                           0
currentLoc                        0
odometer                          0
timestamp                         0
congestionLevel                   0
vehicleID                         0
vehicleLabel                      0
vehicleLicenceplate               0
vehicle_stop_sequence             0
latitude                          0
longitude                         0
bearing                           0
currentStatus                     0
destinationStopID                 0
destinationStopName               0
is_weekend                        0
trip_stop_sequence                0
arrivalDelay                    351
arrivalTime                       0
departureDelay                  360
departureTime                     0
originStopID                      0
tripScheduleRelationship          0
originStopName                    0
dwellTime_sec                     0
is_peak_hour                      0
has_prev_stop                     0
service_date                

In [37]:
merged.to_csv("merged_v5.csv", index=False)

### *VI. Normalizing values*

In [39]:
df = pd.read_csv("merged_v5.csv")

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_36295/11598681.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("merged_v5.csv")


In [40]:
def add_trip_totals(df):
    df = df.copy()
    

    df['travel_time_sec'] = df['travel_time_sec'].astype(float)
    df['distance_m'] = df['distance_m'].astype(float)
    

    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    
    # Create unique trip key
    df['trip_key'] = (
        df['trip_id'].astype(str) + '_' +
        df['date'].astype(str) + '_' +
        df['originStopName'].astype(str) + '_' +
        df['destinationStopName'].astype(str)
    )
    
    # Aggregate total travel time and distance per trip
    trip_totals = df.groupby('trip_key').agg(
        total_travel_time_sec=('travel_time_sec', 'sum'),
        total_distance_m=('distance_m', 'sum')
    ).reset_index()
    
    # Merge back
    df_agg = df.merge(trip_totals, on='trip_key', how='left')
    
    return df_agg

In [41]:
df = add_trip_totals(df)

In [42]:
df = df.drop(columns=["date"])

In [43]:
df = df.drop(columns=["trip_key"])

In [44]:
df.loc[df["trip_id"] == 184, ["travel_time_sec", "total_travel_time_sec"]].head(10)

,travel_time_sec,total_travel_time_sec
0,38.0,174.0
1,53.0,174.0
4,83.0,174.0
8,7.0,230.0
11,22.0,230.0
14,52.0,230.0
18,67.0,230.0
20,82.0,230.0
27,7.0,1248.0
29,22.0,1248.0


In [45]:
df.loc[df["trip_id"] == 184, ["distance_m", "total_distance_m"]].head(10)

,distance_m,total_distance_m
0,529.0,755.0
1,151.0,755.0
4,75.0,755.0
8,35.0,980.0
11,225.0,980.0
14,555.0,980.0
18,146.0,980.0
20,19.0,980.0
27,9.0,2232.0
29,194.0,2232.0


### Normalizing temperature and wind features

- using z-score standardization

In [47]:
zscore_scaler = StandardScaler()
temp_cols = ['temperature_2m','apparent_temperature','windspeed_10m', 'windgusts_10m', 'winddirection_10m']
df[temp_cols] = zscore_scaler.fit_transform(df[temp_cols])

### Normalizing precipitation and rain

- using log and min-max

In [48]:
for col in ['precipitation', 'rain', 'snowfall']:
    df[col] = np.log1p(df[col])

df[['precipitation', 'rain', 'snowfall']] = MinMaxScaler().fit_transform(
    df[['precipitation', 'rain', 'snowfall']]
)

In [12]:
df.head()

,trip_id,currentLoc,odometer,timestamp,congestionLevel,vehicleID,vehicleLabel,vehicleLicenceplate,vehicle_stop_sequence,latitude,...,temperature_2m,apparent_temperature,precipitation,rain,snowfall,windspeed_10m,windgusts_10m,winddirection_10m,total_travel_time_sec,total_distance_m
0,184,POINT (-35.19819641113281 149.14988708496094),156876200,2021-06-01 09:14:45,1,11,LRV11,LRV11,4,-35.198196,...,-1.698996,-1.455002,0.0,0.0,0.0,-1.503387,-1.362655,1.125497,174.0,226.0
1,184,POINT (-35.199607849121094 149.1495819091797),156876351,2021-06-01 09:15:00,1,11,LRV11,LRV11,4,-35.199608,...,-1.698996,-1.455002,0.0,0.0,0.0,-1.503387,-1.362655,1.125497,174.0,226.0
2,49,POINT (-35.258853912353516 149.13238525390625),154385587,2021-06-01 09:15:00,1,9,LRV9,LRV9,5,-35.258854,...,-1.866028,-1.606091,0.0,0.0,0.0,-1.442419,-1.119104,-1.170611,414.0,955.0
3,47,POINT (-35.23988342285156 149.13803100585938),137985437,2021-06-01 09:15:00,1,13,LRV13,LRV13,7,-35.239883,...,-1.866028,-1.606091,0.0,0.0,0.0,-1.442419,-1.119104,-1.170611,335.0,694.0
4,184,POINT (-35.200260162353516 149.1494140625),156876426,2021-06-01 09:15:30,1,11,LRV11,LRV11,4,-35.200260,...,-1.698996,-1.455002,0.0,0.0,0.0,-1.503387,-1.362655,1.125497,174.0,226.0


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5513586 entries, 0 to 5513585
Data columns (total 55 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   trip_id                   object        
 1   currentLoc                object        
 2   odometer                  int64         
 3   timestamp                 datetime64[ns]
 4   congestionLevel           int64         
 5   vehicleID                 int64         
 6   vehicleLabel              object        
 7   vehicleLicenceplate       object        
 8   vehicle_stop_sequence     int64         
 9   latitude                  float64       
 10  longitude                 float64       
 11  bearing                   float64       
 12  currentStatus             int64         
 13  destinationStopID         int64         
 14  destinationStopName       object        
 15  is_weekend                int64         
 16  trip_stop_sequence        int64         
 17  arrivalD

In [50]:
df.to_csv("new_merged_v2.csv", index=False)

## **D. Data Separation**


- 20% Test data

- 60% Training data

- 10% Validation data

- 10% Holdout data

In [6]:
df = pd.read_csv("data/new_merged_v2.csv")

/var/folders/v6/lvzjzwnn7198jjy93qbh54gm0000gn/T/ipykernel_44176/3960192548.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/new_merged_v2.csv")


In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

df = df[
    (df["timestamp"] >= "2021-09-01") &
    (df["timestamp"] <= "2022-09-02")
]

In [7]:
df['arrivalTime'] = pd.to_datetime(df['arrivalTime'])
df['arrivalTime'].min(), df['arrivalTime'].max()

(Timestamp('2021-06-01 09:13:40'), Timestamp('2022-11-24 14:18:02'))

In [8]:
# remove rows with missing critical values for distance, time delta, or speed
cols_to_check = ['distance_m', 'delta_t_sec', 'speed_mps', 'speed_kph']

df_clean = df.dropna(subset=cols_to_check)

print("Original shape:", df.shape)
print("After dropping NA:", df_clean.shape)
print("Rows removed:", df.shape[0] - df_clean.shape[0])

Original shape: (5513586, 55)
After dropping NA: (5434880, 55)
Rows removed: 78706


In [65]:
df_clean.to_csv("new_merged_v3.csv", index=False)

In [9]:
df_merged = df_clean.copy()
df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'], errors='coerce')
df_merged = df_merged.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)

train_prop = 0.6
test_prop  = 0.2
val_prop   = 0.1
hold_prop  = 0.1  # not used directly; remainder becomes holdout

# --- time-based cut points using quantiles of timestamp ---
t1 = df_merged['timestamp'].quantile(train_prop)
t2 = df_merged['timestamp'].quantile(train_prop + test_prop)
t3 = df_merged['timestamp'].quantile(train_prop + test_prop + val_prop)

# --- half-open interval splits to prevent overlap ---
train_df   = df_merged[df_merged['timestamp'] <  t1]
test_df    = df_merged[(df_merged['timestamp'] >= t1) & (df_merged['timestamp'] <  t2)]
val_df     = df_merged[(df_merged['timestamp'] >= t2) & (df_merged['timestamp'] <  t3)]
holdout_df = df_merged[df_merged['timestamp'] >= t3]

print(f"Train:   {train_df['timestamp'].min()} → {train_df['timestamp'].max()}")
print(f"Test:    {test_df['timestamp'].min()} → {test_df['timestamp'].max()}")
print(f"Val:     {val_df['timestamp'].min()} → {val_df['timestamp'].max()}")
print(f"Holdout: {holdout_df['timestamp'].min()} → {holdout_df['timestamp'].max()}")


print("Counts:", len(train_df), len(test_df), len(val_df), len(holdout_df))
print("Overlap check:",
      train_df['timestamp'].max() < test_df['timestamp'].min() if len(test_df) else True,
      test_df['timestamp'].max() < val_df['timestamp'].min() if len(val_df) else True,
      val_df['timestamp'].max() < holdout_df['timestamp'].min() if len(holdout_df) else True)

Train:   2021-06-01 09:14:45 → 2022-05-04 06:55:15
Test:    2022-05-04 06:55:30 → 2022-08-12 10:31:00
Val:     2022-08-12 10:31:30 → 2022-10-07 09:12:45
Holdout: 2022-10-07 09:13:00 → 2022-11-24 14:18:30
Counts: 3260928 1086975 543484 543493
Overlap check: True True True


In [12]:
train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)
val_df.to_csv("validation_data.csv", index=False)
holdout_df.to_csv("holdout_data.csv", index=False)